![JohnSnowLabs](https://sparknlp.org/assets/images/logo.png)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp/blob/master/examples/python/annotation/text/english/embeddings/LateChunkEmbeddings.ipynb)

# Late Chunk Embeddings

Context-aware chunk embeddings for RAG pipelines, based on
[Jin et al. (2024) — *Late Chunking: Contextual Chunk Embeddings Using Long-Context Embedding Models*, arXiv:2409.04701](https://arxiv.org/abs/2409.04701).

## The Problem: Naive Chunking Loses Context

A standard RAG pipeline splits a document into chunks before embedding:

```
Document  ──split──►  Chunk 0  ──embed──►  vector 0
                      Chunk 1  ──embed──►  vector 1
```

Each chunk is embedded in isolation. If chunk 1 depends on something introduced in chunk 0, that signal is lost.

For example:

- Chunk 0: *"AcmeDrug was prescribed for migraine in March. The patient took two doses."*
- Chunk 1: *"It caused severe nausea the next day, and therapy was stopped."*

Chunk 1 is embedded without knowing what "It" refers to. For the query *"Which drug caused severe nausea?"*, the retriever may miss chunk 1 entirely because its isolated embedding never saw the word *AcmeDrug*.

## The Fix: Late Chunking

Late chunking reverses the order:

```
Document ──embed (full doc)──► token vectors [t0…tN] ──pool by span──► chunk vectors
```

1. Run a long-context transformer over the entire document in one forward pass. Every token vector is contextualised by the full document.
2. Apply chunk boundaries after the forward pass.
3. Mean-pool the token vectors inside each chunk span, producing one embedding per chunk.

Paper: Jin et al., arXiv:2409.04701: https://arxiv.org/abs/2409.04701

### Spark NLP contract

`LateChunkEmbeddings` inputs: `DOCUMENT`, `WORD_EMBEDDINGS`, `CHUNK`  
Output: one `SENTENCE_EMBEDDINGS` annotation per chunk.

Note: the token-embedding stage must take `DOCUMENT` (not `SENTENCE`) as input. Placing a `SentenceDetector` before the embedding model defeats late chunking.

## 1: Setup

In [ ]:
!pip install -q pyspark spark-nlp

In [5]:
import sparknlp

spark = sparknlp.start()

print("Spark NLP version: ", sparknlp.version())
print("Apache Spark version: ", spark.version)

Spark NLP version:  6.4.0
Apache Spark version:  3.5.1


## 2: Sample Data

Two chunks where the second sentence depends on context from the first.

In [6]:
document_text = (
    "AcmeDrug was prescribed for migraine in March. The patient took two doses.\n\n"
    "It caused severe nausea the next day, and therapy was stopped."
)

chunk_texts = [
    "AcmeDrug was prescribed for migraine in March. The patient took two doses.",
    "It caused severe nausea the next day, and therapy was stopped.",
]

data = spark.createDataFrame([(document_text, chunk_texts)], ["text", "chunks"])


## 3: Late Chunking Pipeline

In [7]:
from sparknlp.annotator import *
from sparknlp.base import *
from pyspark.ml import Pipeline

document_assembler = DocumentAssembler()\
    .setInputCol("text")\
    .setOutputCol("document")

tokenizer = Tokenizer()\
    .setInputCols(["document"])\
    .setOutputCol("token")

# embed the FULL document in one forward pass
# Input must be 'document' not 'sentence' — preserves cross-sentence context
token_embeddings = (
    LongformerEmbeddings.pretrained("longformer_base_4096", "en")
    .setInputCols(["document", "token"])
    .setOutputCol("token_embeddings")
    .setCaseSensitive(True)
    .setMaxSentenceLength(4096)
)

# chunk boundaries from the pre-split column
chunker = (
    Doc2Chunk()
    .setInputCols(["document"])
    .setChunkCol("chunks")
    .setIsArray(True)
    .setOutputCol("chunk")
)

# pool contextual token vectors per chunk span → SENTENCE_EMBEDDINGS
late_chunk_embeddings = (
    LateChunkEmbeddings()
    .setInputCols(["document", "chunk", "token_embeddings"])
    .setOutputCol("late_chunk_embeddings")
    .setPoolingStrategy("AVERAGE")  # or "SUM"
)

pipeline = Pipeline(stages=[
    document_assembler, tokenizer, token_embeddings, chunker, late_chunk_embeddings
])

model  = pipeline.fit(data)
result = model.transform(data)


longformer_base_4096 download started this may take some time.
Approximate size to download 343.3 MB
[OK!]


## 4: Inspect Output

One SENTENCE_EMBEDDINGS annotation per chunk

In [8]:
result.selectExpr("explode(late_chunk_embeddings) as r") \
    .select("r.annotatorType", "r.result", "r.embeddings") \
    .show(10, truncate=80)


+-------------------+--------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|      annotatorType|                                                                    result|                                                                      embeddings|
+-------------------+--------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|sentence_embeddings|AcmeDrug was prescribed for migraine in March. The patient took two doses.|[0.050471075, -0.075952254, 0.031268872, 0.15105465, -0.013696871, 0.08131724...|
|sentence_embeddings|            It caused severe nausea the next day, and therapy was stopped.|[0.07356851, 0.0060830414, 0.120519586, 0.2239925, 0.05588421, 0.06679503, 0....|
+-------------------+--------------------------------------------------------------------------+--------------

## 5: EmbeddingsFinisher (RAG-ready vectors)

`EmbeddingsFinisher` converts `SENTENCE_EMBEDDINGS` into a plain `Array[Vector]` column ready to upsert into any vector database (Pinecone, Qdrant, Weaviate). For a working example, see the [VectorDBConnector Pinecone demo](https://github.com/JohnSnowLabs/spark-nlp/blob/master/examples/python/annotation/text/english/vector-db/VectorDBConnector_Pinecone_Demo.ipynb) notebook.

In [11]:
finisher = (
    EmbeddingsFinisher()
    .setInputCols(["late_chunk_embeddings"])
    .setOutputCols(["chunk_vectors"])
    .setOutputAsVector(True)
    .setCleanAnnotations(False)
)

finished = finisher.transform(result)
finished.selectExpr("explode(chunk_vectors) as v").show(5, truncate=80)


+--------------------------------------------------------------------------------+
|                                                                               v|
+--------------------------------------------------------------------------------+
|[0.050471074879169464,-0.07595225423574448,0.03126887232065201,0.151054650545...|
|[0.07356850802898407,0.006083041429519653,0.12051958590745926,0.2239924967288...|
+--------------------------------------------------------------------------------+



## Known Limitations

**Context-window cap.** Benefit is bounded by the model's max sequence length (4,096 for Longformer, 8,192 for ModernBERT). Tokens beyond that limit lose cross-chunk context.

**Full-document requirement.** A `SentenceDetector` before the embedding model silently degrades to naive chunking.

**Ordering.** `LateChunkEmbeddings` must appear after the token-embedding stage. Incorrect ordering raises a runtime error.

**Pooling.** Only `AVERAGE` and `SUM` are supported.